In [15]:
import warnings

import numpy as np
from scipy.optimize import minimize

warnings.filterwarnings('ignore')

# Fitting ARMA(p, q)
- Compute Residuals
- Minimise Residuals to for MLE Fit

In [10]:
def arma_residuals(params, ret, p, q, forecast=False):
    '''Compute Residuals of ARMA(p, q) Process'''
    # Initialise vectors
    phi = params[1:p+1]
    theta = params[p+1:]
    n = len(ret)
    resids = np.zeros(n)
    next = np.zeros(2)
    
    # Compute residuals
    for t in range(n):
        add = 0
        
        # AR
        for i in range(1, p+1):
            if t - i >= 0:
                add += phi[i-1] * ret[t-i]
        
        # MA
        for i in range(1, q+1):
            if t - i >= 0:
                add += theta[i-1] * resids[t-i]
                
        if t == n-1 and forecast:  
            next[0] = params[0] + add
            next[1] = np.sqrt(1 / n * sum(resids**2))
        
        resids[t] = ret[t] - params[0] - add
            
    if forecast:
        return resids, next
    else:
        return resids

In [11]:
def arma_likelihood(params, ret, p,  q):
    '''Compute ARMA(p, q) Likelihood through Sum of Square Residuals'''
    resids = arma_residuals(params, ret, p, q)
    return sum(resids**2)

In [17]:
def fit_arma(ret, p, q, forecast=False):
    '''Fit ARMA(p, q) Model to Returns by Minimising Likelihood'''
    # Initial Parameters
    init = np.zeros(p + q + 1)
    init[0] = np.mean(ret)
    
    # Minimise Residuals
    res = minimize(arma_likelihood, x0 = init, args=(ret, p, q,))
    
    if forecast:
        resids, next = arma_residuals(res['x'], ret, p, q, True)
        return res['x'], next
    return res['x']